# IA001 — Análise e Visualização de Dados com Python e Ferramentas Assistidas por IA

## Separar o Rio Grande do Sul pelas Regiões Funcionais de Planejamento

**Objetivo:** unir a malha municipal do IBGE à tabela oficial `municipios_corede_regiao_funcional_rs.csv` e produzir camadas geográficas que separem o território do estado pelas 9 Regiões Funcionais de Planejamento, com os indicadores municipais de 2010 anexados a cada polígono.

**Produto:** um GeoPackage com duas camadas (municípios classificados e polígonos das Regiões Funcionais), um GeoJSON para mapa web, um CSV de atributos, um mapa HTML consultável, com os indicadores de cada município no rótulo do mouse e um registro JSON com as fontes e as decisões de preparação.

**Percurso:** (A) baixar as fontes com registro de procedência; (B) inspecionar e limpar; (C) cruzar **pelo código do IBGE**, verificando a cobertura; (D) anexar os indicadores de 2010; (E) dissolver em regiões; (F) salvar, reabrir e conferir num mapa consultável.

Este roteiro segue o fluxo de `aula03_07_Obtencao_Dados_Geograficos.ipynb`: download com procedência, inspeção antes da transformação, junção validada e reabertura dos arquivos salvos.

## 1. Leia antes de rodar: o que este dado permite

Um mapa por Região Funcional precisa de duas coisas: a **malha** com os polígonos dos municípios e uma **tabela** que diga a que região cada município pertence. Aqui as duas cobrem o estado inteiro.

A tabela é a lista oficial do **Decreto 54.572/2019** (SEPLAG-RS): uma linha por município, com o COREDE e a Região Funcional de Planejamento a que pertence. São 28 COREDEs agrupados em 9 Regiões Funcionais.

| | |
|---|---|
| Municípios do RS na malha do IBGE (2022) | **497** |
| Linhas na tabela oficial | 496 |
| Classificados por junção direta de código | **496** |
| Atribuído por regra explícita (Pinto Bandeira) | **1** |
| Sem região | **0** |

A única lacuna é **Pinto Bandeira**, emancipado de Bento Gonçalves em 2013. A tabela que usamos foi montada para uma análise com dados de 2010, quando o distrito ainda integrava Bento Gonçalves, e por isso não o lista. Como o território está inteiramente dentro do COREDE Serra, o notebook o atribui à mesma região do município de origem, por uma regra nomeada e registrada no JSON de procedência — não por proximidade calculada.

**Consequência para o mapa:** as 9 Regiões Funcionais aparecem como blocos contíguos, e não há área cinza. A extensão territorial de cada região é a oficial.

### O que mudou em relação à versão anterior

Este notebook usava `municipios_regioes_funcionais_rs.csv`, que listava apenas os *principais municípios* de cada COREDE — 84 dos 497, ou 17%. O mapa resultante era majoritariamente cinza e mostrava os **polos**, não as regiões. Aquele arquivo também agrupava os COREDEs em Regiões Funcionais de forma **divergente da oficial**: apenas 3 dos 9 agrupamentos coincidiam com o decreto, e havia uma região "Sudoeste" que não existe. Ambos os problemas desaparecem com a tabela completa.

A junção também mudou de **nome** para **código do IBGE**. Três municípios têm grafia diferente entre a malha e a tabela (`Restinga Sêca`/`Restinga Seca`, `Sant'Ana`/`Santana do Livramento`, `Vespasiano Corrêa`/`Vespasiano Correa`); pelo código, nenhum deles precisa de tratamento especial.

## 2. Preparação e pastas

As saídas ficam em `dados/regioes_funcionais`, criadas a partir da pasta onde o notebook está.

```text
dados/regioes_funcionais/
├── brutos/          # cópias baixadas, com registro de procedência
└── preparados/      # resultados deste roteiro
```

Instale as dependências no mesmo kernel que executa o notebook.

In [ ]:
# Se necessário, descomente:
# %pip install geopandas pandas numpy folium mapclassify

from pathlib import Path
from datetime import datetime, timezone
from urllib.request import urlopen, Request
import hashlib
import json
import re
import unicodedata

import geopandas as gpd
import pandas as pd
import numpy as np
import folium

PASTA_TRABALHO = Path("dados") / "regioes_funcionais"
PASTA_BRUTOS = PASTA_TRABALHO / "brutos"
PASTA_PREPARADOS = PASTA_TRABALHO / "preparados"
for pasta in (PASTA_BRUTOS, PASTA_PREPARADOS):
    pasta.mkdir(parents=True, exist_ok=True)

print("Pasta de trabalho:", PASTA_TRABALHO.resolve())
print("geopandas", gpd.__version__, "| pandas", pd.__version__, "| folium", folium.__version__)

### Download com reutilização e registro de procedência

`baixar` grava os bytes recebidos e registra URL, data UTC, tamanho e SHA-256. O hash identifica a cópia exata usada; não atesta, sozinho, a qualidade do dado. Um arquivo já existente com registro é conferido antes de ser reutilizado, e o download passa por um `.part` para que uma interrupção não deixe um arquivo truncado com o nome final.

In [ ]:
def sha256(arquivo):
    resumo = hashlib.sha256()
    with Path(arquivo).open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            resumo.update(bloco)
    return resumo.hexdigest()


def baixar(url, destino):
    """Baixa uma vez e reutiliza, registrando procedência em <arquivo>.fonte.json."""
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    registro = destino.with_suffix(destino.suffix + ".fonte.json")

    if destino.exists():
        if registro.exists():
            meta = json.loads(registro.read_text(encoding="utf-8"))
            if meta["url"] != url or meta["sha256"] != sha256(destino):
                raise ValueError(f"Cópia local não confere com o registro: {destino}")
        print("Reutilizando:", destino.name)
        return destino

    parcial = destino.with_suffix(destino.suffix + ".part")
    pedido = Request(url, headers={"User-Agent": "IA001-notebook"})
    with urlopen(pedido, timeout=180) as resposta, parcial.open("wb") as saida:
        while bloco := resposta.read(1024 * 1024):
            saida.write(bloco)
    parcial.replace(destino)

    registro.write_text(json.dumps({
        "url": url,
        "baixado_em_utc": datetime.now(timezone.utc).isoformat(),
        "bytes": destino.stat().st_size,
        "sha256": sha256(destino),
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Baixado:", destino.name, f"({destino.stat().st_size / 1_048_576:.1f} MB)")
    return destino

## 3. As fontes

| Fonte | O que traz | Origem |
|---|---|---|
| `RS_Municipios_2022.zip` | Polígonos dos municípios do RS, com código e nome do IBGE | [Malhas municipais, IBGE](https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_municipais/municipio_2022/UFs/RS/) |
| `municipios_corede_regiao_funcional_rs.csv` | Uma linha por município, com código do IBGE, COREDE e Região Funcional | Repositório do grupo, a partir da tabela do [Decreto 54.572/2019 — SEPLAG-RS](https://planejamento.rs.gov.br/coredes) |
| `dataset_principal.csv` | Indicadores do Censo 2010: rendimento dos ocupados, escolaridade e outros | Repositório do grupo, a partir do [Atlas do Desenvolvimento Humano no Brasil](https://www.atlasbrasil.org.br/consulta/planilha) (Pnud, Ipea, FJP) |
| `pib_municipios_rs_2010.csv` | PIB, população e valor adicionado por setor em 2010 | Repositório do grupo, a partir do [SIDRA, tabela 5938](https://sidra.ibge.gov.br/tabela/5938) (IBGE) |

As duas primeiras trazem o **código do IBGE**, e a junção entre elas é feita por código: é a escolha mais segura sempre que existe, porque nome depende de grafia, acento e apóstrofo. O PIB também traz código. O Atlas **não** traz — só o nome do município — e por isso é o único que exige reconciliação de grafia (seção 7).

In [ ]:
BASE_DADOS = ("https://raw.githubusercontent.com/valandro/ufrgs-spec-ai/main/"
              "AI001-analise-visualizacao-dados/educacao_rs/datasets/")

URL_MALHA = (
    "https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/"
    "malhas_municipais/municipio_2022/UFs/RS/RS_Municipios_2022.zip"
)
URL_REGIOES = BASE_DADOS + "municipios_corede_regiao_funcional_rs.csv"
URL_ATLAS = BASE_DADOS + "dataset_principal.csv"
URL_PIB = BASE_DADOS + "pib_municipios_rs_2010.csv"

ARQUIVO_MALHA = baixar(URL_MALHA, PASTA_BRUTOS / "RS_Municipios_2022.zip")
ARQUIVO_REGIOES = baixar(URL_REGIOES, PASTA_BRUTOS / "municipios_corede_regiao_funcional_rs.csv")
ARQUIVO_ATLAS = baixar(URL_ATLAS, PASTA_BRUTOS / "dataset_principal.csv")
ARQUIVO_PIB = baixar(URL_PIB, PASTA_BRUTOS / "pib_municipios_rs_2010.csv")

## 4. Inspecionar a malha antes de transformar

Duas coisas precisam de atenção nesta malha:

1. Ela tem **499 feições, mas o RS tem 497 municípios**. As duas sobrando são corpos d'água — Lagoa dos Patos e Lagoa Mirim — que o IBGE distribui na mesma camada, com códigos reservados `4300001` e `4300002`. Somá-las como municípios inflaria área e contagem.
2. O CRS é `EPSG:4674` (SIRGAS 2000), em graus. **Não é uma projeção métrica**: calcular área diretamente nele daria um número sem significado. A área é calculada mais adiante, em UTM.

Os códigos são preservados como texto: `4300034` não é um número a ser somado.

In [ ]:
malha = gpd.read_file(f"zip://{ARQUIVO_MALHA}")
print("Feições:", len(malha), "| CRS:", malha.crs)
print("Colunas:", malha.columns.tolist())

# Códigos reservados do IBGE para corpos d'água nesta camada — não são municípios.
CODIGOS_AGUA = {"4300001", "4300002"}
malha["CD_MUN"] = malha["CD_MUN"].astype(str)
agua = malha[malha["CD_MUN"].isin(CODIGOS_AGUA)]
print("\nRemovidos por não serem municípios:")
print(agua[["CD_MUN", "NM_MUN", "AREA_KM2"]].to_string(index=False))

municipios = malha[~malha["CD_MUN"].isin(CODIGOS_AGUA)][
    ["CD_MUN", "NM_MUN", "geometry"]
].copy()

if len(municipios) != 497:
    raise ValueError(f"Esperados 497 municípios, encontrados {len(municipios)}.")
if municipios["CD_MUN"].duplicated().any():
    raise ValueError("Há códigos de município repetidos na malha.")
if municipios.geometry.is_empty.any() or not municipios.geometry.is_valid.all():
    raise ValueError("Há geometrias vazias ou inválidas na malha.")

print(f"\nMunicípios válidos: {len(municipios)}")
display(municipios.drop(columns="geometry").head())

## 5. Ler a tabela de Regiões Funcionais

A tabela oficial tem uma linha por município — não é preciso desmembrar nada. O que exige atenção é o **tipo** da coluna de código: `codigo_ibge` precisa ser lido como texto. Como número, `4300034` continua correto por acaso, mas a coluna deixa de casar com a `CD_MUN` da malha, que é texto.

A célula abaixo também confere a integridade da tabela antes de usá-la: códigos únicos, 28 COREDEs, 9 Regiões Funcionais e cada COREDE em uma única Região Funcional.

In [ ]:
regioes = pd.read_csv(ARQUIVO_REGIOES, encoding="utf-8-sig", dtype={"codigo_ibge": str})
print(f"Linhas: {len(regioes)} | COREDEs: {regioes['corede'].nunique()} "
      f"| Regiões Funcionais: {regioes['regiao_funcional_id'].nunique()}")
display(regioes.head(3))

if regioes["codigo_ibge"].duplicated().any():
    raise ValueError("Há código do IBGE repetido na tabela de regiões.")

# Cada COREDE pertence a exatamente uma Região Funcional. Se isso não valer,
# a hierarquia do decreto foi quebrada em algum ponto da preparação do CSV.
por_corede = regioes.groupby("corede")["regiao_funcional_id"].nunique()
if (por_corede > 1).any():
    raise ValueError(f"COREDEs em mais de uma RF: {por_corede[por_corede > 1].index.tolist()}")

print("\nCOREDEs de cada Região Funcional:")
for rf_id, grupo in regioes.groupby("regiao_funcional_id"):
    print(f"  RF{rf_id}  {len(grupo):3d} municípios · {', '.join(sorted(grupo['corede'].unique()))}")

## 6. Cruzar pelo código do IBGE

As duas fontes trazem o código do IBGE, então a junção é direta e não depende de grafia. A validação `m:1` garante que cada município da malha case com no máximo uma linha da tabela.

Resta uma diferença de recorte temporal entre as fontes, e ela precisa de decisão explícita:

- A malha é de **2022** e tem 497 municípios.
- A tabela foi preparada para uma análise de **2010** e tem 496: **Pinto Bandeira** (`4314548`) foi emancipado de Bento Gonçalves em 2013 e ficou de fora.

Deixá-lo sem região abriria um buraco no mapa por uma razão que nada tem a ver com o decreto. Atribuí-lo "ao vizinho mais próximo" seria inventar. A saída usada aqui é a terceira: **herdar a classificação do município de origem**, que é um fato documentado do desmembramento, declarado em `HERDA_DE` e registrado no JSON de procedência. A coluna `origem_rf` marca quais municípios vieram da tabela e qual veio da regra.

In [ ]:
# Municípios criados depois da edição da tabela herdam a classificação do
# município de origem do desmembramento. Chave = código do novo; valor = código
# de origem. Pinto Bandeira (4314548) saiu de Bento Gonçalves (4302105) em 2013.
HERDA_DE = {"4314548": "4302105"}

classificados = municipios.merge(
    regioes[["codigo_ibge", "corede", "regiao_funcional_id"]],
    left_on="CD_MUN", right_on="codigo_ibge", how="left", validate="m:1",
).drop(columns="codigo_ibge")
if len(classificados) != len(municipios):
    raise ValueError("A junção alterou a quantidade de municípios.")

classificados["origem_rf"] = np.where(
    classificados["regiao_funcional_id"].notna(), "tabela oficial", None)

direto = int(classificados["regiao_funcional_id"].notna().sum())
print(f"Classificados pela junção direta: {direto} de {len(classificados)}")

# ---------------------------------------------------------------
# Aplica as heranças declaradas, uma a uma e com registro no print
# ---------------------------------------------------------------
for novo, origem in HERDA_DE.items():
    alvo = classificados["CD_MUN"] == novo
    if not alvo.any():
        continue                     # o município não está nesta malha
    if classificados.loc[alvo, "regiao_funcional_id"].notna().all():
        continue                     # a tabela já o classificou; nada a herdar
    fonte = regioes.loc[regioes["codigo_ibge"] == origem]
    if fonte.empty:
        raise ValueError(f"Origem {origem} não está na tabela; HERDA_DE está desatualizado.")
    fonte = fonte.iloc[0]
    classificados.loc[alvo, ["corede", "regiao_funcional_id"]] = [
        fonte["corede"], fonte["regiao_funcional_id"]]
    classificados.loc[alvo, "origem_rf"] = f"herdado de {fonte['municipio']}"
    print(f"Herança aplicada: {classificados.loc[alvo, 'NM_MUN'].iat[0]} "
          f"← {fonte['municipio']} (COREDE {fonte['corede']}, RF{fonte['regiao_funcional_id']:.0f})")

sem_regiao = classificados[classificados["regiao_funcional_id"].isna()]
if not sem_regiao.empty:
    raise ValueError(
        f"{len(sem_regiao)} municípios sem Região Funcional: "
        f"{sem_regiao['NM_MUN'].tolist()}. Acrescente-os à tabela oficial ou a "
        "HERDA_DE — não os deixe passar em silêncio.")

classificados["regiao_funcional_id"] = classificados["regiao_funcional_id"].astype(int)

# Nomes das 9 RFs, montados a partir dos COREDEs que cada uma reúne.
NOMES_RF = {
    1: "RF1 · Metropolitana", 2: "RF2 · Vale do Rio Pardo e Taquari",
    3: "RF3 · Serra e Hortênsias", 4: "RF4 · Litoral", 5: "RF5 · Sul",
    6: "RF6 · Campanha e Fronteira Oeste", 7: "RF7 · Noroeste e Missões",
    8: "RF8 · Central", 9: "RF9 · Norte e Produção",
}
classificados["regiao_funcional"] = classificados["regiao_funcional_id"].map(NOMES_RF)
if classificados["regiao_funcional"].isna().any():
    raise ValueError("Há Região Funcional sem nome em NOMES_RF.")

print(f"\nCobertura: {len(classificados)} de {len(classificados)} municípios (100%)")
print(classificados["origem_rf"].value_counts().to_string())

### Cobertura por região

Com a tabela completa, esta tabela deixa de ser um diagnóstico de lacuna e passa a ser a **descrição do estado**: quantos municípios e quanta área cada Região Funcional reúne. Os percentuais somam 100%.

A área é calculada em **UTM estimada para o estado**, e não no CRS geográfico original. `estimate_utm_crs` escolhe a zona a partir da extensão dos dados; para um território do tamanho do RS isso é uma aproximação razoável, mas não substitui uma projeção equivalente em área para medições oficiais.

In [ ]:
CRS_AREA = classificados.estimate_utm_crs()
if CRS_AREA is None:
    raise ValueError("Não foi possível estimar a projeção UTM.")
classificados["area_km2"] = classificados.to_crs(CRS_AREA).area / 1_000_000
print("Projeção usada para área:", CRS_AREA.name)

cobertura = (
    classificados
    .groupby("regiao_funcional")
    .agg(municipios=("CD_MUN", "size"),
         coredes=("corede", "nunique"),
         area_km2=("area_km2", "sum"))
    .sort_index()
)
cobertura["% dos municípios"] = 100 * cobertura["municipios"] / len(classificados)
cobertura["% da área do RS"] = 100 * cobertura["area_km2"] / classificados["area_km2"].sum()

print(f"\nAs 9 Regiões Funcionais e os {len(classificados)} municípios do estado:")
display(cobertura.round(1))

soma = cobertura["% da área do RS"].sum()
if abs(soma - 100) > 0.01:
    raise ValueError(f"As regiões deveriam somar 100% da área; somam {soma:.2f}%.")
print(f"Território do RS classificado: {soma:.1f}% da área — o estado inteiro.")

## 7. Acrescentar os indicadores municipais

Até aqui cada município tem região, COREDE e área. Isso basta para desenhar o mapa, mas não para consultá-lo: passar o mouse sobre um município e ler apenas a área é pouco útil quando a pergunta é sobre renda e escolaridade.

Esta seção traz para a camada geográfica os mesmos indicadores usados nos gráficos da atividade 01, para que o mapa responda às mesmas perguntas de forma interativa:

| Indicador | Fonte | Coluna |
|---|---|---|
| Rendimento médio dos ocupados | Atlas / Censo 2010 | `rendimento_ocupados_2010` |
| % dos ocupados com ensino médio completo | Atlas / Censo 2010 | `pct_ocupados_ensino_medio_2010` |
| População | Censo 2010 (IBGE) | `populacao_2010` |
| PIB per capita | PIB-Munic 2010 (IBGE) | `pib_per_capita_2010` |
| Setor de maior valor adicionado | PIB-Munic 2010 (IBGE) | `setor_maior_vab_2010` |

### Duas junções, duas chaves

O PIB traz `codigo_ibge` e entra por código, como a tabela de regiões.

O Atlas **não tem coluna de código** — a identificação é o nome do município seguido de `(RS)`. Aqui não há escolha: a junção é por nome, e três municípios divergem entre as fontes.

| Na tabela de regiões | No Atlas | Resolve com |
|---|---|---|
| `Westfália (RS)` | `Westfalia (RS)` | normalização (acento) |
| `Xangri-lá (RS)` | `Xangri-Lá (RS)` | normalização (caixa e acento) |
| `Santana do Livramento (RS)` | `Sant'Ana do Livramento (RS)` | tabela de equivalências |

Os dois primeiros a normalização resolve sozinha. O terceiro não é variação de grafia, é **nome historicamente diferente**, e por isso vai numa tabela explícita — escondê-lo numa regra genérica de substituição faria a junção passar a depender de um detalhe invisível.

A tabela de regiões serve de **ponte**: tem o código do IBGE e o nome no formato do Atlas, então é ela que liga os dois mundos. É o motivo prático de preferir sempre a fonte que publica código.

### O que fica sem dado

**Pinto Bandeira** não está no Atlas nem no PIB de 2010 — em 2010 era distrito de Bento Gonçalves. Ele mantém a Região Funcional herdada (seção 6), mas os indicadores ficam nulos, e o mapa mostra `sem dado`. Preencher com o valor de Bento Gonçalves seria inventar um município que, naquele ano, não existia como unidade de coleta.

In [ ]:
def normalizar(texto):
    """Sem acento, sem pontuação, minúsculo — para casar nomes entre fontes."""
    texto = unicodedata.normalize("NFKD", str(texto).replace("'", " ").replace("’", " "))
    texto = texto.encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", " ", texto.lower()).strip()


# Nomes historicamente diferentes, que a normalização não reconcilia.
# Chave = nome na tabela de regiões; valor = nome no Atlas.
EQUIVALENCIAS = {"Santana do Livramento (RS)": "Sant'Ana do Livramento (RS)"}

COL_RENDA = "Rendimento médio dos ocupados 2010"
COL_ENSINO = "% dos ocupados com ensino médio completo 2010"

# ---------------------------------------------------------------
# Atlas: junção por nome, via a tabela de regiões como ponte
# ---------------------------------------------------------------
atlas = pd.read_csv(ARQUIVO_ATLAS, usecols=["Territorialidades", COL_RENDA, COL_ENSINO])
# A planilha do Atlas termina com linhas de rodapé (créditos e fontes) que não
# são municípios; elas somem na junção, mas o `to_numeric` precisa tolerá-las.
for coluna in (COL_RENDA, COL_ENSINO):
    atlas[coluna] = pd.to_numeric(atlas[coluna], errors="coerce")
atlas["chave"] = atlas["Territorialidades"].map(normalizar)

ponte = regioes[["codigo_ibge", "Territorialidades"]].copy()
ponte["chave"] = ponte["Territorialidades"].replace(EQUIVALENCIAS).map(normalizar)

sem_par = ponte.loc[~ponte["chave"].isin(atlas["chave"]), "Territorialidades"].tolist()
if sem_par:
    raise ValueError(f"Municípios sem correspondência no Atlas: {sem_par}. "
                     "Acrescente-os a EQUIVALENCIAS em vez de ignorá-los.")

indicadores = ponte.merge(
    atlas[["chave", COL_RENDA, COL_ENSINO]], on="chave", how="left", validate="one_to_one"
).drop(columns=["chave", "Territorialidades"]).rename(columns={
    COL_RENDA: "rendimento_ocupados_2010",
    COL_ENSINO: "pct_ocupados_ensino_medio_2010",
})
print(f"Atlas: {len(indicadores)} municípios cruzados por nome")

# ---------------------------------------------------------------
# PIB: junção por código, e o setor de maior valor adicionado
# ---------------------------------------------------------------
pib = pd.read_csv(ARQUIVO_PIB, dtype={"codigo_ibge": str})

SETORES = {
    "vab_agropecuaria_mil_reais": "Agropecuária",
    "vab_industria_mil_reais": "Indústria",
    "vab_servicos_exceto_adm_publica_mil_reais": "Serviços",
    "vab_administracao_publica_mil_reais": "Administração pública",
}
pib["setor_maior_vab_2010"] = pib[list(SETORES)].idxmax(axis=1).map(SETORES)

indicadores = indicadores.merge(
    pib[["codigo_ibge", "populacao_censo_2010", "pib_per_capita_2010", "setor_maior_vab_2010"]],
    on="codigo_ibge", how="left", validate="one_to_one",
).rename(columns={"populacao_censo_2010": "populacao_2010"})
print(f"PIB:   {indicadores['pib_per_capita_2010'].notna().sum()} municípios cruzados por código")

# ---------------------------------------------------------------
# Trazer para a camada geográfica
# ---------------------------------------------------------------
classificados = classificados.merge(
    indicadores, left_on="CD_MUN", right_on="codigo_ibge", how="left", validate="one_to_one"
).drop(columns="codigo_ibge")

COLUNAS_INDICADORES = ["rendimento_ocupados_2010", "pct_ocupados_ensino_medio_2010",
                       "populacao_2010", "pib_per_capita_2010", "setor_maior_vab_2010"]

print(f"\nCobertura dos indicadores, em {len(classificados)} municípios:")
for coluna in COLUNAS_INDICADORES:
    faltando = classificados.loc[classificados[coluna].isna(), "NM_MUN"].tolist()
    print(f"  {coluna:32s} {len(classificados) - len(faltando):3d} com dado"
          + (f" · sem: {', '.join(faltando)}" if faltando else ""))

print("\nSetor de maior valor adicionado:")
print(classificados["setor_maior_vab_2010"].value_counts().to_string())

## 8. Dissolver: de municípios para regiões

`dissolve` funde os polígonos dos municípios de cada região num único registro. Como agora todos os municípios estão classificados, cada Região Funcional vira um **bloco contíguo** — e não o conjunto de manchas soltas que a cobertura parcial produzia.

A coluna `partes` conta os pedaços desconexos de cada região, e a coluna `maior_parte_%` diz quanto do território está no maior deles. As duas juntas separam dois casos muito diferentes:

- **Ilhas.** RF1 e RF4 aparecem com mais de uma parte por causa de ilhas de fração de km² no Delta do Jacuí e no litoral. O maior pedaço concentra praticamente 100% da área.
- **Separação real por água.** A RF5 · Sul é cortada pela Lagoa dos Patos, removida da malha na seção 4: a margem leste (São José do Norte e vizinhos) fica desconectada do restante. É uma descontinuidade do território, não do processamento.

Em nenhum dos casos `partes > 1` indica erro de geometria.

In [ ]:
regioes_geo = (
    classificados
    .dissolve(by="regiao_funcional", aggfunc={"CD_MUN": "count", "area_km2": "sum"})
    .rename(columns={"CD_MUN": "municipios"})
    .reset_index()
)


def partes_de(geometria):
    """Quantos pedaços desconexos, e que fração da área está no maior deles."""
    pedacos = list(geometria.geoms) if geometria.geom_type == "MultiPolygon" else [geometria]
    areas = sorted((p.area for p in pedacos), reverse=True)
    return len(pedacos), 100 * areas[0] / sum(areas)


em_metros = regioes_geo.to_crs(CRS_AREA)
regioes_geo[["partes", "maior_parte_%"]] = [partes_de(g) for g in em_metros.geometry]
# a atribuição em bloco promove tudo a float; a contagem de pedaços é inteira
regioes_geo["partes"] = regioes_geo["partes"].astype(int)

if len(regioes_geo) != 9:
    raise ValueError(f"Esperadas 9 Regiões Funcionais, obtidas {len(regioes_geo)}.")
if regioes_geo["municipios"].sum() != len(classificados):
    raise ValueError("A dissolução perdeu municípios pelo caminho.")

print(f"Regiões Funcionais representadas: {len(regioes_geo)} de 9")
display(regioes_geo.drop(columns="geometry").round(2))

fragmentadas = regioes_geo[regioes_geo["maior_parte_%"] < 99]
print("\nRegiões com pedaço secundário relevante (mais de 1% da área):")
print(fragmentadas[["regiao_funcional", "partes", "maior_parte_%"]].round(1).to_string(index=False)
      if not fragmentadas.empty else "  nenhuma")


## 9. Salvar os dados preparados

O GeoPackage guarda as duas camadas — municípios classificados e regiões dissolvidas — no mesmo arquivo, sem simplificar a geometria. O CSV traz só os atributos. O GeoJSON vai em `EPSG:4326` (longitude/latitude) e com simplificação de 200 m, adequada à escala estadual e suficiente para reduzir o peso do mapa web.

`to_crs` **transforma** coordenadas; `set_crs` apenas declara o sistema de referência. Trocar o rótulo não reprojeta nada.

In [ ]:
SAIDA_GPKG = PASTA_PREPARADOS / "regioes_funcionais_rs.gpkg"
SAIDA_CSV = PASTA_PREPARADOS / "municipios_classificados.csv"
SAIDA_GEOJSON = PASTA_PREPARADOS / "municipios_web.geojson"

colunas_saida = (["CD_MUN", "NM_MUN", "regiao_funcional", "regiao_funcional_id",
                  "corede", "origem_rf", "area_km2"]
                 + COLUNAS_INDICADORES + ["geometry"])
municipios_saida = classificados[colunas_saida]

# mode="a" ACRESCENTA linhas a uma camada existente. Sem apagar o arquivo antes,
# rodar o notebook duas vezes duplicaria a camada de regiões. Apagar e reescrever
# mantém a célula idempotente.
SAIDA_GPKG.unlink(missing_ok=True)
municipios_saida.to_file(SAIDA_GPKG, layer="municipios", driver="GPKG", mode="w", index=False)
regioes_geo.to_file(SAIDA_GPKG, layer="regioes_funcionais", driver="GPKG", mode="a", index=False)
municipios_saida.drop(columns="geometry").to_csv(SAIDA_CSV, index=False, encoding="utf-8")

web = municipios_saida.to_crs(CRS_AREA).copy()
web["geometry"] = web.geometry.simplify(200, preserve_topology=True)
web = web.to_crs(4326)
web.to_file(SAIDA_GEOJSON, driver="GeoJSON", index=False)

for arquivo in (SAIDA_GPKG, SAIDA_CSV, SAIDA_GEOJSON):
    print(f"{arquivo.name:34s} {arquivo.stat().st_size / 1_048_576:6.2f} MB")

### Registrar fontes e decisões

Um nome de arquivo não diz qual edição foi usada nem que escolhas foram feitas no caminho. O registro abaixo acompanha os dados e guarda o que um leitor precisaria para reproduzir ou contestar o resultado — inclusive a base legal da tabela, a chave usada na junção e exatamente qual município foi classificado por herança em vez de pela fonte.

In [ ]:
def descrever_fonte(arquivo, url):
    registro = arquivo.with_suffix(arquivo.suffix + ".fonte.json")
    meta = json.loads(registro.read_text(encoding="utf-8")) if registro.exists() else {}
    return {"arquivo": arquivo.name, "url": url, "sha256": sha256(arquivo),
            "bytes": arquivo.stat().st_size, "baixado_em_utc": meta.get("baixado_em_utc")}


heranca = classificados.loc[
    classificados["origem_rf"] != "tabela oficial", ["CD_MUN", "NM_MUN", "origem_rf"]]

metadados = {
    "objetivo": "Separar os municípios do RS pelas Regiões Funcionais de Planejamento",
    "preparado_em_utc": datetime.now(timezone.utc).isoformat(),
    "malha": {"ano": 2022, "crs_original": str(malha.crs), "crs_area": str(CRS_AREA)},
    "tabela_de_regioes": {
        "base_legal": "Decreto 54.572/2019 (SEPLAG-RS)",
        "linhas": int(len(regioes)),
        "coredes": int(regioes["corede"].nunique()),
        "regioes_funcionais": int(regioes["regiao_funcional_id"].nunique()),
    },
    "cobertura": {
        "municipios_no_estado": int(len(classificados)),
        "com_regiao_funcional": int(len(classificados)),
        "sem_regiao_funcional": 0,
        "classificados_pela_tabela": int((classificados["origem_rf"] == "tabela oficial").sum()),
        "classificados_por_heranca": int(len(heranca)),
        "percentual_da_area_classificada": 100.0,
    },
    "indicadores": {
        "colunas": COLUNAS_INDICADORES,
        "ano_de_referencia": 2010,
        "sem_dado": classificados.loc[
            classificados["rendimento_ocupados_2010"].isna(), "NM_MUN"].tolist(),
    },
    "decisoes": [
        "Removidas as feições 4300001 e 4300002 (Lagoa Mirim e Lagoa dos Patos): não são municípios.",
        "Junção pelo código do IBGE (CD_MUN × codigo_ibge), e não por nome: independe de grafia.",
        "Conferido que cada COREDE pertence a uma única Região Funcional.",
        ("Municípios ausentes da tabela herdam a classificação do município de origem do "
         f"desmembramento, conforme HERDA_DE = {HERDA_DE}. Aplicado a: "
         f"{heranca['NM_MUN'].tolist() or 'nenhum'}."),
        "Nenhum município foi atribuído por proximidade geográfica.",
        ("Indicadores do Atlas cruzados por nome normalizado, pois a fonte não "
         f"publica código do IBGE. Equivalências aplicadas: {EQUIVALENCIAS}."),
        "Indicadores de PIB e população cruzados pelo código do IBGE.",
        ("Municípios sem dado em 2010 permanecem nulos; nenhum valor foi imputado "
         "a partir do município de origem."),
    ],
    "fontes": [
        descrever_fonte(ARQUIVO_MALHA, URL_MALHA),
        descrever_fonte(ARQUIVO_REGIOES, URL_REGIOES),
    ],
}

SAIDA_META = PASTA_PREPARADOS / "fontes_e_preparo.json"
SAIDA_META.write_text(json.dumps(metadados, ensure_ascii=False, indent=2), encoding="utf-8")
print(SAIDA_META.read_text(encoding="utf-8")[:1100], "...")

## 10. Reabrir e conferir

Salvar sem testar a leitura esconde problemas de coluna, codificação e formato. Reabrimos os três arquivos e comparamos contagem, códigos, área e CRS com o que está em memória.

O CSV não carrega esquema de tipos: ao reabrir, o código do município precisa ser declarado como texto, ou `4300034` volta como número.

In [ ]:
mun_reaberto = gpd.read_file(SAIDA_GPKG, layer="municipios")
rf_reaberto = gpd.read_file(SAIDA_GPKG, layer="regioes_funcionais")
csv_reaberto = pd.read_csv(SAIDA_CSV, dtype={"CD_MUN": str})
web_reaberto = gpd.read_file(SAIDA_GEOJSON)

assert len(mun_reaberto) == len(csv_reaberto) == len(web_reaberto) == len(classificados)
assert set(mun_reaberto["CD_MUN"]) == set(classificados["CD_MUN"])
assert len(rf_reaberto) == len(regioes_geo) == 9
assert mun_reaberto["regiao_funcional"].notna().all(), "Município sem região após reabrir."
assert mun_reaberto.crs == classificados.crs
assert web_reaberto.crs.to_epsg() == 4326
assert abs(mun_reaberto["area_km2"].sum() - classificados["area_km2"].sum()) < 1
for coluna in COLUNAS_INDICADORES:
    assert coluna in mun_reaberto.columns, f"Indicador perdido na gravação: {coluna}"
assert (mun_reaberto["rendimento_ocupados_2010"].notna().sum()
        == classificados["rendimento_ocupados_2010"].notna().sum())

print("Leitura validada: GPKG (2 camadas), CSV e GeoJSON.")
print(f"  municípios: {len(mun_reaberto)} | regiões: {len(rf_reaberto)} | sem região: 0")
com_renda = int(mun_reaberto["rendimento_ocupados_2010"].notna().sum())
print(f"  indicadores de 2010: {com_renda} municípios com dado, "
      f"{len(mun_reaberto) - com_renda} sem")
print(f"  área total: {mun_reaberto['area_km2'].sum():,.0f} km²".replace(",", "."))

## 11. Conferência visual

O mapa é montado a partir do **GeoJSON reaberto**, o que demonstra que outro notebook pode consumir a saída sem refazer download nem junção.

Leitura do mapa: cada cor é uma Região Funcional, e o estado inteiro está colorido. A linha branca fina separa municípios; a linha escura, desenhada por cima a partir da camada dissolvida, marca a fronteira entre as regiões.

### O que o mouse mostra

Passar o cursor sobre um município abre os indicadores de 2010 acrescentados na seção 7 — rendimento dos ocupados, escolaridade, população, PIB per capita e setor econômico dominante. É o que transforma o mapa de ilustração em ferramenta de consulta: dá para comparar dois municípios vizinhos sem sair da figura, ou procurar o caso que destoa da cor da sua região.

Os números são formatados no padrão brasileiro **apenas para exibição**. As colunas gravadas no GeoPackage e no CSV continuam numéricas, porque texto formatado não se soma nem se ordena. Onde não há dado, o mapa escreve `sem dado` em vez de `NaN` ou de um zero enganoso.

### O mapa abre enquadrado no estado

`fit_bounds` calcula o zoom a partir do tamanho do contêiner. No HTML salvo, esse cálculo às vezes roda antes de o `div` ter altura, e sem tamanho o Leaflet devolve zoom 0: o mundo inteiro, com o Rio Grande do Sul reduzido a um ponto. A célula repete o enquadramento depois do evento `load`, com `invalidateSize()` antes, para que a abertura não dependa de quando o navegador terminou o layout.

### Por que o mapa-base não é o da CARTO

A versão anterior usava `tiles="CartoDB positron"`. O `xyzservices`, biblioteca que o `geopandas` e o `folium` consultam para localizar mapas-base, passou a marcar os mosaicos da CARTO como **`requires_token: True`**: a empresa migrou os basemaps gratuitos para um modelo com chave de API vinculada a uma conta.

Na prática, hoje, isso depende de como se pede o mosaico:

- `folium.Map(tiles="CartoDB positron")` — o atalho embutido do folium monta a URL sozinho, sem parâmetro `key`, e os mosaicos **ainda respondem**;
- `folium.Map(tiles=xyzservices.providers.CartoDB.Positron)` ou `GeoDataFrame.explore()` — passam pelo `xyzservices`, e é aí que a chave entra na URL. Sem informá-la, o endereço sai com o texto de exemplo `<insert your API key here>` e os mosaicos falham em silêncio, deixando o fundo branco.

Depender de um serviço que já anunciou o fechamento é um risco desnecessário num material que precisa rodar daqui a meses, na máquina de outra pessoa. O mapa usa o **OpenStreetMap**, que não pede chave nem conta. Como os polígonos são opacos, o mapa-base só aparece fora do estado, e a troca não muda a leitura.

In [ ]:
# Paleta qualitativa: as Regiões Funcionais não têm ordem, então cores
# distinguíveis entre si valem mais que um gradiente.
CORES_RF = {
    "RF1 · Metropolitana": "#e6550d",
    "RF2 · Vale do Rio Pardo e Taquari": "#31a354",
    "RF3 · Serra e Hortênsias": "#3182bd",
    "RF4 · Litoral": "#756bb1",
    "RF5 · Sul": "#e7ba52",
    "RF6 · Campanha e Fronteira Oeste": "#17becf",
    "RF7 · Noroeste e Missões": "#d6616b",
    "RF8 · Central": "#8c6d31",
    "RF9 · Norte e Produção": "#c994c7",
}
faltando = set(web_reaberto["regiao_funcional"]) - set(CORES_RF)
if faltando:
    raise ValueError(f"Regiões sem cor definida: {faltando}")

SEM_DADO = "sem dado"


def formatar(valor, casas=0, prefixo="", sufixo=""):
    """Número no padrão brasileiro: ponto no milhar, vírgula no decimal.

    A troca é feita só sobre o número, nunca sobre prefixo e sufixo: aplicada
    à string inteira, o ponto de "hab." viraria vírgula.

    O marcador temporário \x00 impede que a segunda substituição desfaça a
    primeira — sem ele, vírgula e ponto acabariam ambos como ponto.
    """
    if pd.isna(valor):
        return SEM_DADO
    numero = f"{valor:,.{casas}f}".replace(",", "\x00").replace(".", ",").replace("\x00", ".")
    return f"{prefixo}{numero}{sufixo}"


# Colunas só de exibição. As numéricas seguem intactas no GPKG e no CSV —
# texto formatado não se soma nem se ordena.
rotulos = web_reaberto.copy()
rotulos["t_renda"] = rotulos["rendimento_ocupados_2010"].map(
    lambda v: formatar(v, 0, prefixo="R$ "))
rotulos["t_ensino"] = rotulos["pct_ocupados_ensino_medio_2010"].map(
    lambda v: formatar(v, 1, sufixo="%"))
rotulos["t_pop"] = rotulos["populacao_2010"].map(
    lambda v: formatar(v, 0, sufixo=" hab."))
rotulos["t_pib"] = rotulos["pib_per_capita_2010"].map(
    lambda v: formatar(v, 2, prefixo="R$ "))
rotulos["t_setor"] = rotulos["setor_maior_vab_2010"].fillna(SEM_DADO)

CAMPOS_TOOLTIP = {
    "NM_MUN": "Município:",
    "regiao_funcional": "Região Funcional:",
    "corede": "COREDE:",
    "t_renda": "Rendimento médio dos ocupados:",
    "t_ensino": "Ocupados com ensino médio:",
    "t_pop": "População (Censo 2010):",
    "t_pib": "PIB per capita (2010):",
    "t_setor": "Maior valor adicionado:",
}


def estilo(feicao):
    regiao = feicao["properties"]["regiao_funcional"]
    return {"fillColor": CORES_RF[regiao], "color": "white",
            "weight": 0.4, "fillOpacity": 0.85}


# OpenStreetMap: sem chave de API e sem conta — ver a nota da seção 11 sobre
# a CARTO. O mapa-base só aparece fora do estado, já que os polígonos cobrem
# todo o RS de forma opaca.
mapa = folium.Map(tiles="OpenStreetMap", control_scale=True, height=620)

folium.GeoJson(
    rotulos.__geo_interface__, name="Municípios", style_function=estilo,
    # Realce ao passar o mouse: sem ele, num mapa denso, é fácil ler o rótulo
    # de um município enquanto se olha para o vizinho.
    highlight_function=lambda _: {"weight": 2.2, "color": "#1a1a1a", "fillOpacity": 0.95},
    tooltip=folium.GeoJsonTooltip(
        fields=list(CAMPOS_TOOLTIP), aliases=list(CAMPOS_TOOLTIP.values()),
        sticky=True, labels=True,
        style=("background-color: white; border: 1px solid #bbb; border-radius: 3px; "
               "padding: 6px; font: 12px sans-serif;")),
).add_to(mapa)

# Fronteiras entre regiões, por cima dos municípios: sem elas, a linha branca
# de divisa municipal e a divisa regional ficam indistinguíveis.
fronteiras = rf_reaberto.to_crs(CRS_AREA)
fronteiras["geometry"] = fronteiras.geometry.simplify(200, preserve_topology=True)
folium.GeoJson(
    fronteiras.to_crs(4326).__geo_interface__, name="Fronteiras das regiões",
    style_function=lambda _: {"fillOpacity": 0, "color": "#2b2b2b", "weight": 1.6},
    interactive=False,
).add_to(mapa)

folium.LayerControl(collapsed=False).add_to(mapa)

# ---------------------------------------------------------------
# Enquadrar o estado
# ---------------------------------------------------------------
# `fit_bounds` sozinho não basta. O Leaflet deriva o zoom do tamanho do
# contêiner, e no HTML salvo esse cálculo às vezes roda antes de o div ter
# altura: sem tamanho, ele devolve zoom 0 — o mundo inteiro, com o RS
# reduzido a um ponto. Repetir o enquadramento depois do evento `load`,
# precedido de `invalidateSize()` para o Leaflet remedir o contêiner,
# torna a abertura determinística.
oeste, sul, leste, norte = web_reaberto.total_bounds
LIMITES = [[float(sul), float(oeste)], [float(norte), float(leste)]]
mapa.fit_bounds(LIMITES)
mapa.get_root().script.add_child(folium.Element(
    f"window.addEventListener('load', function () {{"
    f"  var alvo = {mapa.get_name()};"
    f"  alvo.invalidateSize();"
    f"  alvo.fitBounds({json.dumps(LIMITES)});"
    f"}});"
))

contagem = web_reaberto["regiao_funcional"].value_counts()
legenda = "".join(
    f'<div><span style="background:{cor};width:12px;height:12px;'
    f'display:inline-block;margin-right:6px"></span>{nome} '
    f'<span style="color:#777">({contagem[nome]})</span></div>'
    for nome, cor in CORES_RF.items()
)
mapa.get_root().html.add_child(folium.Element(
    '<div style="position:fixed;bottom:24px;left:24px;z-index:9999;background:white;'
    'padding:10px 12px;border:1px solid #ccc;border-radius:4px;font:12px sans-serif">'
    '<b>Regiões Funcionais de Planejamento — RS</b>'
    f'<div style="color:#777;margin-bottom:4px">{len(web_reaberto)} municípios · '
    'Decreto 54.572/2019</div>' + legenda +
    '<div style="color:#777;margin-top:6px">(n) = municípios na região<br>'
    'Passe o mouse para ver os indicadores de 2010</div></div>'
))

# Salva o mapa como HTML autocontido, para abrir no navegador sem rodar o
# notebook de novo. O arquivo precisa de internet para o mapa-base e para as
# bibliotecas JavaScript do Leaflet.
SAIDA_MAPA = PASTA_PREPARADOS / "mapa_regioes_funcionais.html"
mapa.save(SAIDA_MAPA)
print("Mapa salvo em:", SAIDA_MAPA.resolve())
print(f"  {SAIDA_MAPA.stat().st_size / 1_048_576:.1f} MB — abra com duplo clique ou arraste para o navegador")

mapa

## 12. O que fazer com isto

### Como visualizar

| Quero... | Como |
|---|---|
| Ver o mapa sem rodar nada | Abrir `dados/regioes_funcionais/preparados/mapa_regioes_funcionais.html` no navegador |
| Ver o mapa aqui no notebook | Executar a seção 11 — o mapa aparece embaixo da célula |
| Inspecionar as geometrias | Arrastar `municipios_web.geojson` para [geojson.io](https://geojson.io) |
| Abrir num SIG | `regioes_funcionais_rs.gpkg` no QGIS; as duas camadas aparecem na lista |
| Só os atributos | `municipios_classificados.csv` em qualquer planilha |

O HTML é autocontido quanto aos dados, mas busca o mapa-base e as bibliotecas do Leaflet na internet — sem conexão, os polígonos aparecem sobre fundo branco. No Jupyter, se o mapa não renderizar, marque o notebook como confiável (*Trust Notebook*).

### Continuar a análise

Os produtos estão em `dados/regioes_funcionais/preparados/`. Para continuar em outro notebook:

```python
import geopandas as gpd
municipios = gpd.read_file(
    "dados/regioes_funcionais/preparados/regioes_funcionais_rs.gpkg",
    layer="municipios"
)
regioes = gpd.read_file(
    "dados/regioes_funcionais/preparados/regioes_funcionais_rs.gpkg",
    layer="regioes_funcionais"
)
```

As camadas já trazem os indicadores de 2010 anexados pela seção 7:

| Coluna | Conteúdo |
|---|---|
| `rendimento_ocupados_2010` | Rendimento médio dos ocupados, em reais |
| `pct_ocupados_ensino_medio_2010` | % dos ocupados com ensino médio completo |
| `populacao_2010` | População do Censo 2010 |
| `pib_per_capita_2010` | PIB per capita, em reais |
| `setor_maior_vab_2010` | Setor de maior valor adicionado |
| `origem_rf` | Se a Região Funcional veio da tabela oficial ou de herança |

Para acrescentar outros indicadores, use `CD_MUN` como chave sempre que a outra fonte tiver código do IBGE — é mais seguro que nome, como os três casos de grafia da seção 7 mostram. O `pib_municipios_rs_2010.csv` traz o código; o `dataset_principal.csv` não, e se liga pela coluna `Territorialidades`, que é o nome do município seguido de `(RS)`.

### Limitações

- **Recorte temporal misto.** A malha é de 2022 e a tabela de regiões foi preparada para uma análise de 2010. A consequência prática é Pinto Bandeira, resolvido por herança declarada (seção 6). Se a composição dos COREDEs mudar por decreto posterior, a diferença não é detectável aqui.
- **Pinto Bandeira classificado por regra, não pela fonte, e sem indicadores.** A coluna `origem_rf` permite filtrá-lo. Ele não aparece no Censo 2010 nem no PIB-Munic 2010 porque era distrito de Bento Gonçalves, e o notebook não imputa valores: as colunas ficam nulas e o mapa escreve `sem dado`. Para uma análise de 2010, o correto é somá-lo a Bento Gonçalves, e não tratá-lo como município à parte.
- **Indicadores de 2010 sobre malha de 2022.** Os polígonos são atuais; os números, do Censo de 2010. Para os 496 municípios que existiam nas duas datas isso é irrelevante, mas convém lembrar ao comparar áreas com densidades.
- **`setor_maior_vab_2010` é o máximo, não a maioria.** Um município onde os serviços respondem por 30% do valor adicionado aparece como "Serviços" mesmo sem que o setor seja dominante. Para medir concentração, calcule a participação de cada setor a partir das colunas de VAB do arquivo de PIB.
- **Área em UTM estimada.** Adequada para comparação relativa; para medida oficial, use a `AREA_KM2` publicada pelo IBGE na própria malha.
- **A geometria do GeoJSON é simplificada em 200 m.** Serve à escala estadual do mapa web. Para cálculo de área ou vizinhança, use a camada do GeoPackage, que não é simplificada.
- **Regiões Funcionais não são unidades de coleta.** Nenhum dado é publicado nativamente nessa escala: todo indicador por região é uma agregação de municípios, sujeita a efeito de composição e à falácia ecológica.

### Exercícios

1. Compare a `area_km2` calculada com a coluna `AREA_KM2` da malha original. As diferenças vêm da projeção escolhida — quantifique-as.
2. Junte o `pib_municipios_rs_2010.csv` e calcule o PIB per capita mediano de cada Região Funcional. Depois calcule o PIB per capita **da região** (soma do PIB dividida pela soma da população) e explique por que os dois números diferem.
3. Dissolva por COREDE em vez de por Região Funcional e compare: quanto da variação entre regiões se mantém no recorte mais fino?
4. Refaça o mapa com `tiles=None` e sem mapa-base. O que se perde e o que se ganha em legibilidade?
5. Troque a cor do mapa: em vez de uma cor por Região Funcional, faça um coroplético de `rendimento_ocupados_2010` em quintis, mantendo as fronteiras regionais por cima. As regiões continuam legíveis como blocos de renda?
6. Acrescente ao rótulo a posição do município dentro da sua região (`rank` do rendimento). O que isso revela sobre a heterogeneidade interna da RF1?